<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/01_hello_openimpala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 1: The Hook -- Zero to Physics in 60 Seconds

**The Pain Point:** Setting up physical simulations on 3D microstructures usually takes days of meshing, configuring HPC environments, and compiling C++. By the time you get a result, you've forgotten the scientific question you were trying to ask.

**The Solution:** OpenImpala is a massively parallel, exascale solver that installs in seconds via `pip` and eats raw NumPy arrays for breakfast. No meshes. No Makefiles. Just physics.

**In this tutorial we will:**
1. Generate a synthetic 3D gyroid microstructure in pure NumPy (no external files).
2. Calculate volume fraction, percolation, and tortuosity with one function call each.
3. Run a parametric sweep of porosity vs. tortuosity -- the building block for autonomous, AI-driven inverse design.

In [ ]:
# Install OpenImpala and plotting libraries
# (This takes < 5 seconds thanks to pre-compiled static wheels!)
!pip install -q openimpala matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded and ready.")

# --- THE JUPYTER HPC FIX ---
# We instantiate a global session and enter it.
# This keeps AMReX and MPI alive for the entire duration of the notebook,
# preventing kernel crashes when running multiple cells!
global_session = oi.Session()
global_session.__enter__()

## 1. Generating a Synthetic Microstructure

To prove OpenImpala doesn't rely on pre-formatted specific files, we will generate a **gyroid** surface from scratch. Gyroids are triply-periodic minimal surfaces widely used as scaffold geometries in energy materials.

We will generate a 1 million voxel ($100^3$) grid.
- **Phase 0** (solid matrix)
- **Phase 1** (void / pore space)

In [ ]:
def generate_gyroid(size=100, porosity_target=0.40, frequency=4.0):
    """Generate a binary 3D gyroid microstructure using pure NumPy."""
    lin = np.linspace(0, 2 * np.pi * frequency, size, endpoint=False)
    x, y, z = np.meshgrid(lin, lin, lin, indexing="ij")

    # Gyroid level-set function
    field = np.sin(x) * np.cos(y) + np.sin(y) * np.cos(z) + np.sin(z) * np.cos(x)

    # Choose threshold so that the desired fraction of voxels are above it
    threshold = np.percentile(field, (1.0 - porosity_target) * 100.0)

    # Phase 0 = solid, Phase 1 = void (pore)
    binary = (field > threshold).astype(np.int32)
    return binary

# Generate 1,000,000 voxels
N = 100
micro = generate_gyroid(size=N, porosity_target=0.40)

print(f"Microstructure shape: {micro.shape} ({micro.size / 1e6:.1f} million voxels)")
print(f"Measured porosity:    {micro.mean():.4f}")

Let's look at the mid-plane cross-section of our generated structure. Black = solid, white = pore.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(micro[N // 2, :, :], cmap="gray", interpolation="nearest")
ax.set_title(f"Gyroid mid-plane slice (z = {N // 2})")
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. The OpenImpala Handoff

Now we pass our standard NumPy array directly into OpenImpala. Behind the scenes, the highly-optimized C++ bindings will:
1. Seamlessly ingest the NumPy array directly into an AMReX `iMultiFab` in milliseconds.
2. Formulate the steady-state diffusion Laplacian.
3. Solve the resulting system using HYPRE's algebraic multigrid solvers.

*Note: We pass `max_grid_size=128` because our domain ($100^3$) fits in memory. This prevents AMReX from needlessly chopping the domain into tiny blocks, letting the solver run at maximum speed on a laptop/Colab.*

In [ ]:
print("Starting physics calculations...")

# 1. Volume fraction
vf = oi.volume_fraction(micro, phase=1)
print(f"\nPorosity (phase 1 VF): {vf.fraction:.4f}")

# 2. Percolation check (Does the pore space span the domain?)
t0 = time.time()
perc = oi.percolation_check(micro, phase=1, direction="z")
print(f"Percolates in Z:       {perc.percolates} (Found in {time.time()-t0:.3f}s)")

# 3. Solve for Tortuosity
if perc.percolates:
    t0 = time.time()
    result = oi.tortuosity(micro, phase=1, direction="z", solver="flexgmres", max_grid_size=128)
    t_solve = time.time() - t0

    print(f"\nTortuosity (tau):      {result.tortuosity:.4f}")
    print(f"Solver iterations:     {result.iterations} (Residual: {result.residual_norm:.2e})")

    # --- THE SPEED FLEX ---
    print(f"\nSolved {micro.size / 1e6:.1f} million voxels in {t_solve:.2f} seconds.")
else:
    print("Phase does not percolate -- tortuosity is undefined.")

## 3. Parametric Sweep: Tortuosity vs. Porosity

Because OpenImpala is incredibly fast and operates natively on Python arrays, scripting parametric sweeps is trivial.

Below we sweep over a range of target porosities, generate a gyroid for each, and map the tortuosity curve. This capability is the foundational building block for **autonomous, AI-driven inverse design** loops (see [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb)).

In [ ]:
porosities = [0.35, 0.45, 0.55, 0.65, 0.75]
results = []  # (porosity, tau) pairs

print("Running parametric sweep...")
sweep_start = time.time()

for target in porosities:
    # Generate a smaller grid (64^3) for the sweep to keep it snappy
    data = generate_gyroid(size=64, porosity_target=target)

    perc = oi.percolation_check(data, phase=1, direction="z")
    if perc.percolates:
        res = oi.tortuosity(data, phase=1, direction="z", solver="flexgmres", max_grid_size=128)
        results.append((target, res.tortuosity))
        print(f"  Target Porosity={target:.2f}  ->  Tau={res.tortuosity:.4f}")

print(f"\nEvaluated {len(porosities)} microstructures in {time.time() - sweep_start:.2f} seconds.")

# --- Plot the highly publishable sweep ---
if results:
    phi_vals, tau_vals = zip(*results)

    fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
    ax.plot(phi_vals, tau_vals, "o-", color="#d62728", linewidth=2.5, markersize=8,
            markerfacecolor="white", markeredgewidth=2)

    # Styling for publishable look
    ax.set_xlabel(r"Porosity ($\varepsilon$)", fontsize=12, fontweight='bold')
    ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontsize=12, fontweight='bold')
    ax.set_title("Gyroid: Transport Efficiency vs. Porosity", fontsize=14, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

## What's Next?

You just generated a microstructure, solved PDEs, and ran a parametric sweep -- all from a single Python session. No meshes, no input files, no compilation.

**Continue the journey:**
- [Tutorial 2: The Digital Twin](02_digital_twin.ipynb) -- Load a real CT scan, measure anisotropy, and export to PyBaMM's BPX format.
- [Tutorial 3: REV & Uncertainty](03_rev_and_uncertainty.ipynb) -- Prove your results are statistically representative.
- [Tutorial 5: AI Data Factory](05_surrogate_modelling.ipynb) -- Use OpenImpala to generate training data for ML surrogates.
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) -- Put OpenImpala inside a generative design loop.